In [1]:
import os
import sys
import pandas as pd

PARQUET_DIR = "/Users/yuliia/Documents/Fraud-Detection/parquet"
PROCESSED_PATH = os.path.join(PARQUET_DIR, "processed_transactions.parquet")
WAITER_WEEK_FEATURES_PATH = os.path.join(PARQUET_DIR, "waiter_week_features.parquet")
CLIENT_FEATURES_PATH = os.path.join(PARQUET_DIR, "client_level_features.parquet")

ROOT = os.path.dirname(PARQUET_DIR)
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)


def _fraud_waiter_ids() -> set:
    fraud_waiter_ids = set()
    if os.path.exists(CLIENT_FEATURES_PATH):
        cd = pd.read_parquet(CLIENT_FEATURES_PATH)
        if "is_fraud" in cd.columns and "top_waiter_id" in cd.columns:
            fraud_waiter_ids = set(
                cd.loc[cd["is_fraud"] == 1, "top_waiter_id"].dropna().unique()
            )
    if not fraud_waiter_ids:
        from config import FRAUD_WAITER_IDS

        fraud_waiter_ids = set(FRAUD_WAITER_IDS)
    return fraud_waiter_ids


def build_waiter_week_features(df: pd.DataFrame) -> pd.DataFrame:
    """Per waiter_week aggregates (models/waiter_week.ipynb logic)."""
    from config import FRAUD_IDS

    df = df.copy()
    fraud_waiter_ids = _fraud_waiter_ids()
    df["fraud_person_id"] = df["person_id"].isin(FRAUD_IDS)
    df["fraud_waiter_id"] = df["waiter_id"].isin(fraud_waiter_ids)

    ww_fraud = (
        df.groupby(["waiter_id", "week"], as_index=False)
        .agg(
            _fw=("fraud_waiter_id", "first"),
            _fp=("fraud_person_id", "any"),
        )
        .assign(fraud_waiter_week=lambda x: x["_fw"] & x["_fp"])
        .drop(columns=["_fw", "_fp"])
    )
    df = df.merge(ww_fraud, on=["waiter_id", "week"], how="left")

    waiter_week_data = df.groupby("waiter_week").agg(
        num_of_trn=("trn_id", "nunique"),
        unique_persons=("person_id", "nunique"),
        is_fraud=("fraud_waiter_week", "any"),
        working_days=("date", "nunique"),
        place_id=("place_id", "first"),
        week=("week", "first"),
        bonusses_accum=("bonusses_accum", "sum"),
        bonusses_used=("bonusses_used", "sum"),
        bonusses_trn=("bonus_used_flag", "sum"),
        mean_check=("gross_amount", "mean"),
        new_clients=("trn_count_before", lambda x: (x == 0).sum()),
    ).reset_index()

    waiter_week_data["trn_per_person"] = (
        waiter_week_data["num_of_trn"] / waiter_week_data["unique_persons"]
    )
    waiter_week_data["trn_per_person_norm"] = (
        waiter_week_data["trn_per_person"]
        / waiter_week_data.groupby(["place_id", "week"])["trn_per_person"].transform(
            "median"
        )
    )

    waiter_week_data["bonusses_used_norm_w"] = (
        waiter_week_data["bonusses_used"]
        / waiter_week_data.groupby(["place_id", "week"])["bonusses_used"].transform(
            "median"
        )
    )
    waiter_week_data["bonusses_used_norm_l"] = (
        waiter_week_data["bonusses_used"]
        / waiter_week_data.groupby(["place_id"])["bonusses_used"].transform("median")
    )

    waiter_week_data["share_bonusses_trn"] = (
        waiter_week_data["bonusses_trn"] / waiter_week_data["num_of_trn"]
    )
    waiter_week_data["share_bonusses_trn_norm"] = (
        waiter_week_data["share_bonusses_trn"]
        / waiter_week_data.groupby(["place_id", "week"])["share_bonusses_trn"].transform(
            "median"
        )
    )

    waiter_week_data["bonuses_used_to_accum"] = (
        waiter_week_data["bonusses_used"] / waiter_week_data["bonusses_accum"]
    )
    waiter_week_data["bonuses_used_to_accum_norm"] = (
        waiter_week_data["bonuses_used_to_accum"]
        / waiter_week_data.groupby(["place_id", "week"])[
            "bonuses_used_to_accum"
        ].transform("median")
    )

    waiter_week_data["mean_check_norm"] = (
        waiter_week_data["mean_check"]
        / waiter_week_data.groupby(["place_id", "week"])["mean_check"].transform(
            "median"
        )
    )

    waiter_week_data["share_new_clients"] = (
        waiter_week_data["new_clients"] / waiter_week_data["unique_persons"]
    )
    waiter_week_data["share_new_clients_norm"] = (
        waiter_week_data["share_new_clients"]
        / waiter_week_data.groupby(["place_id", "week"])["share_new_clients"].transform(
            "median"
        )
    )
    waiter_week_data["new_clients_norm"] = (
        waiter_week_data["new_clients"]
        / waiter_week_data.groupby(["place_id", "week"])["new_clients"].transform(
            "median"
        )
    )

    waiter_week_data["trn_per_day"] = (
        waiter_week_data["num_of_trn"] / waiter_week_data["working_days"]
    )
    waiter_week_data["trn_per_day_norm"] = (
        waiter_week_data["trn_per_day"]
        / waiter_week_data.groupby(["place_id", "week"])["trn_per_day"].transform(
            "median"
        )
    )

    waiter_week_data["unique_clients_per_day"] = (
        waiter_week_data["unique_persons"] / waiter_week_data["working_days"]
    )
    waiter_week_data["unique_clients_per_day_norm"] = (
        waiter_week_data["unique_clients_per_day"]
        / waiter_week_data.groupby(["place_id", "week"])[
            "unique_clients_per_day"
        ].transform("median")
    )

    client_trn = (
        df.groupby(["waiter_week", "person_id"])
        .agg(trn_per_client=("trn_id", "nunique"))
        .reset_index()
    )
    top_client = (
        client_trn.groupby("waiter_week")["trn_per_client"]
        .max()
        .reset_index()
        .rename(columns={"trn_per_client": "top1_client_trn"})
    )
    waiter_week_data = waiter_week_data.merge(top_client, on="waiter_week", how="left")

    waiter_week_data["top1_client_share"] = (
        waiter_week_data["top1_client_trn"] / waiter_week_data["num_of_trn"]
    )
    waiter_week_data["top1_client_share_norm"] = (
        waiter_week_data["top1_client_share"]
        / waiter_week_data.groupby(["place_id", "week"])["top1_client_share"].transform(
            "median"
        )
    )

    place_week = df.groupby(["place_id", "week"]).agg(
        place_num_of_waiters=("waiter_id", "nunique"),
        place_num_of_clients=("person_id", "nunique"),
        place_num_of_trn=("trn_id", "nunique"),
        total_working_days=("date", "count"),
    )
    waiter_week_data = waiter_week_data.merge(
        place_week, on=["place_id", "week"], how="left"
    )

    waiter_week_data["share_of_clients"] = (
        waiter_week_data["unique_persons"] / waiter_week_data["place_num_of_clients"]
    )
    waiter_week_data["share_of_trn"] = (
        waiter_week_data["num_of_trn"] / waiter_week_data["place_num_of_trn"]
    )
    waiter_week_data["expected_share_of_trn"] = (
        waiter_week_data["working_days"] / waiter_week_data["total_working_days"]
    )
    waiter_week_data["diff_share_of_trn"] = (
        waiter_week_data["share_of_trn"] - waiter_week_data["expected_share_of_trn"]
    )
    waiter_week_data['share_unique_clients'] = waiter_week_data['unique_clients_per_day'] / waiter_week_data['num_of_trn']

    return waiter_week_data


df = pd.read_parquet(PROCESSED_PATH, engine="pyarrow")

if os.path.exists(WAITER_WEEK_FEATURES_PATH):
    waiter_week_data = pd.read_parquet(WAITER_WEEK_FEATURES_PATH, engine="pyarrow")
else:
    waiter_week_data = build_waiter_week_features(df)
    waiter_week_data.to_parquet(
        WAITER_WEEK_FEATURES_PATH, index=False, engine="pyarrow"
    )